# Section 1: Split and baseline

In [1]:
import sys
sys.path.append('../src')

from data import load_hour_data, chronological_split, get_xy

In [2]:
df = load_hour_data()

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (17379, 17)

First 5 rows:


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [3]:
train, test = chronological_split(df)

print("Training rows:", len(train))
print("Testing rows:", len(test))

print("\nTraining period:")
print("From:", train['dteday'].min())
print("To:", train['dteday'].max())

print("\nTesting period:")
print("From:", test['dteday'].min())
print("To:", test['dteday'].max())

Training rows: 13003
Testing rows: 4376

Training period:
From: 2011-01-01 00:00:00
To: 2012-06-30 00:00:00

Testing period:
From: 2012-07-01 00:00:00
To: 2012-12-31 00:00:00


### Why use a chronological split?

This dataset contains bike rental observations over time, so the model should learn from past data and predict future data. A random split could place future observations in the training set while earlier observations are in the test set, which would not represent a real forecasting situation. Therefore, the data is split chronologically at 2012-07-01, with earlier observations used for training and later observations used for testing.

In [4]:
X_train, y_train = get_xy(train)
X_test, y_test = get_xy(test)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nX_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (13003, 12)
y_train shape: (13003,)

X_test shape: (4376, 12)
y_test shape: (4376,)


### Baseline Linear Regression

For the baseline model, I use four simple features: hour of day (`hr`), temperature (`temp`), whether it is a working day (`workingday`), and year (`yr`). This gives us a simple starting model that later models can be compared against.

In [5]:
baseline_features = ['hr', 'temp', 'workingday', 'yr']

X_train_baseline = X_train[baseline_features]
X_test_baseline = X_test[baseline_features]

display(X_train_baseline.head())

,hr,temp,workingday,yr
0,0,0.24,0,0
1,1,0.22,0,0
2,2,0.22,0,0
3,3,0.24,0,0
4,4,0.24,0,0


In [6]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
from sklearn.linear_model import LinearRegression

baseline_model = LinearRegression()

baseline_model.fit(X_train_baseline, y_train)

baseline_predictions = baseline_model.predict(X_test_baseline)

print("Baseline model trained successfully.")
print("Number of predictions:", len(baseline_predictions))

Baseline model trained successfully.
Number of predictions: 4376


In [7]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
mae = mean_absolute_error(y_test, baseline_predictions)
r2 = r2_score(y_test, baseline_predictions)

print("Baseline Linear Regression Results")
print("----------------------------------")
print("RMSE:", rmse)
print("MAE :", mae)
print("R2  :", r2)

Baseline Linear Regression Results
----------------------------------
RMSE: 191.29796121072496
MAE : 145.32068901300278
R2  : 0.24178549551517592


### Random Split Demo — NOT the headline result

The chronological split above is the main evaluation because this is time-based data. The following random split is only a demonstration to compare how the evaluation changes when observations from different time periods are mixed.

In [8]:
from sklearn.model_selection import train_test_split

X_random = X_train[baseline_features]
y_random = y_train

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X_random,
    y_random,
    test_size=0.3,
    random_state=42
)

random_model = LinearRegression()
random_model.fit(X_train_random, y_train_random)

random_predictions = random_model.predict(X_test_random)

random_rmse = np.sqrt(
    mean_squared_error(y_test_random, random_predictions)
)
random_mae = mean_absolute_error(y_test_random, random_predictions)
random_r2 = r2_score(y_test_random, random_predictions)

print("Random Split Demo Results")
print("-------------------------")
print("RMSE:", random_rmse)
print("MAE :", random_mae)
print("R2  :", random_r2)

Random Split Demo Results
-------------------------
RMSE: 128.22980430638523
MAE : 97.1755913113558
R2  : 0.3460524185547861


### Chronological vs Random Split

The random split gives better scores than the chronological split. The random split has an RMSE of about 128.23 compared with 191.30 for the chronological split, while its R² is about 0.346 compared with 0.242.

However, the chronological split is the main result because this dataset is time-based. A random split mixes observations from different time periods, so the model can learn from data that would not be available when making a real future prediction. Therefore, the random split can make the model performance look better than it would in a realistic forecasting situation.

In [9]:
import pandas as pd
from baselines import persistence_forecast

# "Same hour last week" reference forecast (past data only, see src/baselines.py)
persistence_predictions = persistence_forecast(df, train, test)

baseline_results = pd.DataFrame({
    'dteday': test['dteday'].values,
    'hr': test['hr'].values,
    'y_true': y_test.values,
    'baseline': baseline_predictions,
    'persistence': persistence_predictions
})

baseline_results.to_csv(
    '../results/preds_baseline.csv',
    index=False
)

display(baseline_results.head())

print("Saved results/preds_baseline.csv successfully.")

,dteday,hr,y_true,baseline,persistence
0,2012-07-01,0,149,218.190681,163.0
1,2012-07-01,1,93,219.785255,127.0
2,2012-07-01,2,90,221.379829,114.0
3,2012-07-01,3,33,229.374799,60.0
4,2012-07-01,4,4,230.969373,13.0


Saved results/preds_baseline.csv successfully.


### Persistence baseline: same hour last week

The linear baseline above uses four features. A second, even simpler reference is to forecast every hour with the demand of the same hour one week earlier. It needs no training, and it already captures the daily and weekly pattern. A machine-learning model is only useful if it beats it. (If the value one week earlier is missing from the data, the forecast falls back to the training-period average for that hour and day type.)

In [10]:
persistence_rmse = np.sqrt(mean_squared_error(y_test, persistence_predictions))
persistence_mae = mean_absolute_error(y_test, persistence_predictions)
persistence_r2 = r2_score(y_test, persistence_predictions)
baseline_rmse = rmse

print("Persistence (same hour last week)")
print("---------------------------------")
print("RMSE:", persistence_rmse)
print("MAE :", persistence_mae)
print("R2  :", persistence_r2)

Persistence (same hour last week)
---------------------------------
RMSE: 107.88068424044245
MAE : 63.09933401438069
R2  : 0.7588658115475971


### Limitation of the Baseline Model

The baseline Linear Regression model is simple and cannot capture all the patterns in bike demand. In particular, it treats `hr` as a normal numerical feature and assumes a simple linear relationship between hour and demand. However, bike demand can have multiple peaks during a day, such as morning and evening commuting hours. A single linear relationship with `hr` cannot represent these two peaks properly. This is one reason why the baseline model has relatively high prediction error.

# Section 2: Linear, Ridge, Lasso and Random Forest

In this section, we train and compare four regression models. The linear models (Linear, Ridge, Lasso) use `build_preprocessor('linear')` (one-hot hour x working day) and the random forest uses `build_preprocessor('forest')` (cyclic hour), see `src/preprocessing.py` and Section 1.6 of `02-preprocessing.ipynb`. Hyperparameters are selected using TimeSeriesSplit on the training data only, while the test set is reserved for final evaluation.

## 2.1 Linear Regression

First, we train a Linear Regression model using the linear preprocessing pipeline. The preprocessing is fitted only on the training data to avoid data leakage. We then evaluate the fitted model on the held-out test period using RMSE, MAE and R².

In [11]:
# Import required libraries
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import sys
import os
import numpy as np

# Add src folder to Python path
sys.path.append(os.path.abspath("../src"))

import pandas as pd
from preprocessing import build_preprocessor

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# Linear Regression Pipeline (linear preprocessing: one-hot hour x working day)
linear_pipe = Pipeline([
    ('prep', build_preprocessor('linear')),
    ('model', LinearRegression())
])

# Train model
linear_pipe.fit(X_train, y_train)

# Prediction
linear_pred = linear_pipe.predict(X_test)

# Evaluation
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_mae = mean_absolute_error(y_test, linear_pred)
linear_r2 = r2_score(y_test, linear_pred)

print(f"Linear Regression RMSE: {linear_rmse:.2f}")
print(f"Linear Regression MAE:  {linear_mae:.2f}")
print(f"Linear Regression R2:   {linear_r2:.3f}")

Linear Regression RMSE: 107.09
Linear Regression MAE:  80.20
Linear Regression R2:   0.762


## 2.2 Ridge Regression

Ridge Regression adds L2 regularisation to Linear Regression, which helps control large coefficients and can improve stability when features are correlated.

The regularisation strength (`alpha`) is selected using GridSearchCV with TimeSeriesSplit on the training data only. We do not choose alpha using the test set because the test period must remain unseen until final evaluation. TimeSeriesSplit is used instead of shuffled K-fold cross-validation because this is time-ordered data and future observations should not be used to predict earlier observations.

In [12]:
# =========================
# RIDGE REGRESSION
# =========================

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# Time Series Cross Validation
tscv = TimeSeriesSplit(n_splits=5)

# Ridge Pipeline
ridge_pipe = Pipeline([
    ('prep', build_preprocessor('linear')),
    ('model', Ridge())
])

# Alpha values to test
ridge_param_grid = {
    'model__alpha': np.logspace(-3, 3, 13)
}

# Grid Search
ridge_grid = GridSearchCV(
    ridge_pipe,
    ridge_param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=1
)

# Train
ridge_grid.fit(X_train, y_train)

# Best parameter
print("Best Ridge alpha:", ridge_grid.best_params_['model__alpha'])
_g = ridge_param_grid['model__alpha']
print("Best alpha at the edge of the grid?", ridge_grid.best_params_['model__alpha'] in (_g[0], _g[-1]))
# Best Ridge model
ridge_best = ridge_grid.best_estimator_

# Prediction
ridge_pred = ridge_best.predict(X_test)

# Evaluation
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_r2 = r2_score(y_test, ridge_pred)

print(f"Ridge Regression RMSE: {ridge_rmse:.2f}")
print(f"Ridge Regression MAE:  {ridge_mae:.2f}")
print(f"Ridge Regression R2:   {ridge_r2:.3f}")

Best Ridge alpha: 0.001
Best alpha at the edge of the grid? True
Ridge Regression RMSE: 107.09
Ridge Regression MAE:  80.20
Ridge Regression R2:   0.762


## 2.3 Lasso Regression

Lasso Regression uses L1 regularisation. Unlike Ridge, Lasso can shrink some coefficients exactly to zero, so it can also perform a form of feature selection.

The regularisation strength is selected using GridSearchCV with the same TimeSeriesSplit on the training data only.

In [13]:
# =========================
# LASSO REGRESSION
# =========================

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Time Series Cross Validation
tscv = TimeSeriesSplit(n_splits=5)

# Lasso Pipeline
lasso_pipe = Pipeline([
    ('prep', build_preprocessor('linear')),
    ('model', Lasso(max_iter=200000))
])

# Hyperparameter grid
lasso_param_grid = {
    'model__alpha': np.logspace(-4, 1, 12)
}

# Grid Search
lasso_grid = GridSearchCV(
    lasso_pipe,
    lasso_param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=1
)

# Train
lasso_grid.fit(X_train, y_train)

# Best alpha
print("Best Lasso alpha:", lasso_grid.best_params_['model__alpha'])
_g = lasso_param_grid['model__alpha']
print("Best alpha at the edge of the grid?", lasso_grid.best_params_['model__alpha'] in (_g[0], _g[-1]))

# Best Lasso model
lasso_best = lasso_grid.best_estimator_

# Prediction
lasso_pred = lasso_best.predict(X_test)

# Evaluation
lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso_pred))
lasso_mae = mean_absolute_error(y_test, lasso_pred)
lasso_r2 = r2_score(y_test, lasso_pred)

print(f"Lasso Regression RMSE: {lasso_rmse:.2f}")
print(f"Lasso Regression MAE:  {lasso_mae:.2f}")
print(f"Lasso Regression R2:   {lasso_r2:.3f}")

Best Lasso alpha: 0.0001
Best alpha at the edge of the grid? True
Lasso Regression RMSE: 107.09
Lasso Regression MAE:  80.20
Lasso Regression R2:   0.762


In [14]:
# Get feature names after preprocessing
lasso_feature_names = lasso_best.named_steps['prep'].get_feature_names_out()

# Get Lasso coefficients
lasso_coefficients = lasso_best.named_steps['model'].coef_

# Find coefficients that became exactly zero
zero_mask = lasso_coefficients == 0
zero_features = lasso_feature_names[zero_mask]

print("Total coefficients:", len(lasso_coefficients))
print("Number of coefficients exactly zero:", zero_mask.sum())

print("\nFeatures with zero coefficients:")
for feature in zero_features:
    print(feature)

Total coefficients: 70
Number of coefficients exactly zero: 3

Features with zero coefficients:
hour_daytype__hr_wd_15_1
cat__weekday_0
cat__weekday_4


## 2.4 Random Forest Regression

Random Forest is a non-linear ensemble model that combines predictions from multiple decision trees. Unlike the linear models, it can naturally capture non-linear relationships and interactions between features, such as the relationship between hour of day and working-day status.

The model is tuned using TimeSeriesSplit on the training data only. We compare the number of trees, the minimum leaf size and the share of features tried at each split (`max_features`). The first version of this notebook also searched `max_depth` (10, 20, None) and the best value was `None` (no limit), so the depth is left unlimited here.

In [15]:
# =========================
# RANDOM FOREST REGRESSION
# =========================

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Time Series Cross Validation
tscv = TimeSeriesSplit(n_splits=5)

# Random Forest Pipeline (trees use the 'forest' preprocessing: cyclic hour)
forest_pipe = Pipeline([
    ('prep', build_preprocessor('forest')),
    ('model', RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

# Hyperparameter grid.
# max_depth=None and min_samples_leaf=1 are already the least-restricted settings,
# so the grid now also searches the number of trees (up to 400) and max_features.
forest_param_grid = {
    'model__n_estimators': [200, 400],
    'model__max_depth': [None],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': [0.5, 1.0]
}

# Grid Search
forest_grid = GridSearchCV(
    forest_pipe,
    forest_param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=1
)

# Train
forest_grid.fit(X_train, y_train)

# Best parameters
print("Best Random Forest parameters:")
print(forest_grid.best_params_)

# Best model
forest_best = forest_grid.best_estimator_

# Prediction
forest_pred = forest_best.predict(X_test)

# Evaluation
forest_rmse = np.sqrt(mean_squared_error(y_test, forest_pred))
forest_mae = mean_absolute_error(y_test, forest_pred)
forest_r2 = r2_score(y_test, forest_pred)

print(f"\nRandom Forest RMSE: {forest_rmse:.2f}")
print(f"Random Forest MAE:  {forest_mae:.2f}")
print(f"Random Forest R2:   {forest_r2:.3f}")

# Cross-validation RMSE of every setting, to see how flat the grid is
cv_table = (pd.DataFrame(forest_grid.cv_results_)[['param_model__n_estimators', 'param_model__min_samples_leaf',
                                                    'param_model__max_features', 'mean_test_score']]
            .assign(cv_rmse=lambda d: -d['mean_test_score']).drop(columns='mean_test_score')
            .sort_values('cv_rmse').reset_index(drop=True))
display(cv_table.round(2))

Best Random Forest parameters:
{'model__max_depth': None, 'model__max_features': 1.0, 'model__min_samples_leaf': 1, 'model__n_estimators': 400}



Random Forest RMSE: 83.46
Random Forest MAE:  56.26
Random Forest R2:   0.856


,param_model__n_estimators,param_model__min_samples_leaf,param_model__max_features,cv_rmse
0,400,1,1.0,81.21
1,200,1,1.0,81.36
2,400,2,1.0,81.92
3,200,2,1.0,82.00
4,200,1,0.5,82.14
5,400,1,0.5,82.15
6,400,2,0.5,83.36
7,200,2,0.5,83.54
8,400,4,1.0,83.69
9,200,4,1.0,83.81


## 2.5 Save Model Predictions and Best Parameters

The predictions from all four models are saved for the evaluation stage (the baseline and persistence predictions were saved in Section 1). The selected hyperparameters are also saved so that the final evaluation can reproduce the chosen models.

In [16]:
preds_models = pd.DataFrame({
    'dteday': test['dteday'].values,
    'hr': test['hr'].values,
    'y_true': y_test.values,
    'linear': linear_pred,
    'ridge': ridge_pred,
    'lasso': lasso_pred,
    'forest': forest_pred
})

preds_models.to_csv('../results/preds_models.csv', index=False)

print("Saved ../results/preds_models.csv")
print("Shape:", preds_models.shape)

preds_models.head()

Saved ../results/preds_models.csv
Shape: (4376, 7)


,dteday,hr,y_true,linear,ridge,lasso,forest
0,2012-07-01,0,149,213.703747,213.717003,213.710439,141.4550
1,2012-07-01,1,93,184.322664,184.335555,184.326668,109.8725
2,2012-07-01,2,90,167.914099,167.972862,167.920147,76.7550
3,2012-07-01,3,33,138.393774,138.467250,138.399204,35.6050
4,2012-07-01,4,4,120.433294,120.519389,120.439319,9.2550


In [17]:
# =========================
# SAVE BEST PARAMETERS
# =========================

import pandas as pd

best_params = pd.DataFrame([
    {
        'model': 'ridge',
        'param': 'alpha',
        'value': str(ridge_grid.best_params_['model__alpha'])
    },
    {
        'model': 'lasso',
        'param': 'alpha',
        'value': str(lasso_grid.best_params_['model__alpha'])
    },
    {
        'model': 'forest',
        'param': 'n_estimators',
        'value': str(forest_grid.best_params_['model__n_estimators'])
    },
    {
        'model': 'forest',
        'param': 'max_depth',
        'value': str(forest_grid.best_params_['model__max_depth'])
    },
    {
        'model': 'forest',
        'param': 'min_samples_leaf',
        'value': str(forest_grid.best_params_['model__min_samples_leaf'])
    },
    {
        'model': 'forest',
        'param': 'max_features',
        'value': str(forest_grid.best_params_['model__max_features'])
    }
])

# Save CSV
best_params.to_csv('../results/best_params.csv', index=False)

print("Saved ../results/best_params.csv")

best_params

Saved ../results/best_params.csv


,model,param,value
0,ridge,alpha,0.001
1,lasso,alpha,0.0001
2,forest,n_estimators,400
3,forest,max_depth,None
4,forest,min_samples_leaf,1
5,forest,max_features,1.0


## 2.6 Final Model Comparison

All models are compared below using RMSE on the same held-out test period, including the two reference forecasts from Section 1 (the 4-feature baseline and the "same hour last week" persistence forecast). Lower RMSE indicates better predictive accuracy.

In [18]:
model_comparison = pd.DataFrame({
    'Model': [
        'Baseline (4 features, linear)',
        'Persistence (same hour last week)',
        'Linear Regression',
        'Ridge Regression',
        'Lasso Regression',
        'Random Forest'
    ],
    'Test RMSE': [
        baseline_rmse,
        persistence_rmse,
        linear_rmse,
        ridge_rmse,
        lasso_rmse,
        forest_rmse
    ]
})

model_comparison = model_comparison.sort_values('Test RMSE').reset_index(drop=True)

model_comparison

,Model,Test RMSE
0,Random Forest,83.462886
1,Linear Regression,107.090411
2,Ridge Regression,107.091838
3,Lasso Regression,107.092711
4,Persistence (same hour last week),107.880684
5,"Baseline (4 features, linear)",191.297961


### Model Comparison Findings

**Random Forest is the best model.** Test RMSE is about 83.46 bikes per hour (MAE 56.3, R² 0.856). Its best settings were 400 trees, no depth limit, `min_samples_leaf=1` and `max_features=1.0`. The cross-validation table above is flat (best 81.2, and 82.0 for `min_samples_leaf=2`; 200 versus 400 trees changes it by only 0.15), so more tuning or more trees would not help much.

**The linear models improved a lot with the new preprocessing.** Linear, Ridge and Lasso now have a test RMSE of about 107.1 (R² 0.762), compared with 165.7 in the first version (one-hot hour x working day instead of cyclic hour, see `02-preprocessing.ipynb` section 1.6). The three are practically identical. Ridge and Lasso still pick the smallest alpha in their grids (0.001 and 0.0001). This is not a grid problem: with 13,003 training rows and 70 columns the model does not overfit, so cross-validation prefers almost no penalty, and a smaller alpha would just reproduce ordinary least squares. Lasso sets 3 of the 70 coefficients exactly to zero (`hr_wd_15_1`, `weekday_0`, `weekday_4`).

**Reference forecasts.** The persistence forecast (same hour last week, no training) has RMSE 107.9 and MAE 63.1. So the linear models only match it in RMSE and are worse in MAE (80.2). Only the random forest clearly beats it: RMSE is about 23% lower (83.5 versus 107.9) and MAE about 11% lower (56.3 versus 63.1). The 4-feature baseline (RMSE 191.3) is far behind both.

Therefore, among the models tested here, Random Forest is the strongest for predicting hourly bike-rental demand on the held-out test period, and persistence is the bar any model has to clear.